In [2]:
import os
import sys
import getpass
from pyspark.sql import SparkSession

# 1. Kunci IP lokal agar tidak bingung dengan IP Wi-Fi
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 2. Kunci runtime Java 11 & Python interpreter aktif
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 3. Buat SparkSession
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark-ArchLinux") \
    .master("local[*]") \
    .getOrCreate()

# 4. Reduksi pesan diagnostik agar hanya menampilkan ERROR penting
spark.sparkContext.setLogLevel("ERROR")

print("=== STATUS SPARK SESSION ===")
print("Spark Master :", spark.sparkContext.master)
print("Versi Spark  :", spark.version)
print("Status       : SparkSession siap digunakan!")

=== STATUS SPARK SESSION ===
Spark Master : local[*]
Versi Spark  : 3.5.9
Status       : SparkSession siap digunakan!


In [3]:
import os

if not os.path.exists("data_transaksi_ecommerce.csv"):
    import numpy as np
    import pandas as pd
    
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    tanggal_range = pd.date_range("2026-07-01", "2026-07-31", freq="D")
    
    data = {
        "order_id": [f"ORD-{1000+i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "kota": np.random.choice(kota_list, size=n),
        "unit_terjual": np.random.randint(1, 10, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000, 500000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n, p=[0.35, 0.30, 0.20, 0.15]),
    }
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset 'data_transaksi_ecommerce.csv' berhasil dibuat (600 baris).")
else:
    print("Dataset 'data_transaksi_ecommerce.csv' sudah tersedia.")

Dataset 'data_transaksi_ecommerce.csv' berhasil dibuat (600 baris).


In [6]:
# Membaca berkas CSV lokal ke dalam Spark DataFrame
# header=True      -> Menganggap baris pertama sebagai header kolom
# inferSchema=True  -> Spark otomatis mendeteksi tipe data kolom (integer, string, double)
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)

print("Tipe Objek Python:", type(df))
print("--- SKEMA STRUKTUR DATA (printSchema) ---")
df.printSchema()

Tipe Objek Python: <class 'pyspark.sql.dataframe.DataFrame'>
--- SKEMA STRUKTUR DATA (printSchema) ---
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)



In [7]:
# Menampilkan 5 baris pertama (setara dengan df.head() pada Pandas)
df.show(5)

# Menghitung total jumlah baris data (setara dengan len(df) pada Pandas)
print("Total Jumlah Baris Transaksi:", df.count())

+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Bank|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
only showing top 5 rows

Total Jumlah

In [8]:
df.select("order_id", "kota", "kategori").show(5)

+--------+----------+--------------------+
|order_id|      kota|            kategori|
+--------+----------+--------------------+
|ORD-1000|  Magelang|Kesehatan & Kecan...|
|ORD-1001|Yogyakarta|   Makanan & Minuman|
|ORD-1002| Purworejo|             Fashion|
|ORD-1003|      Solo|          Elektronik|
|ORD-1004|  Semarang|          Elektronik|
+--------+----------+--------------------+
only showing top 5 rows



In [9]:
from pyspark.sql.functions import col

df_filtered = df.filter((col("kategori") == "Elektronik") & (col("unit_terjual") > 5))
print("Jumlah transaksi Elektronik dengan unit > 5:", df_filtered.count())
df_filtered.show(5)

Jumlah transaksi Elektronik dengan unit > 5: 65
+--------+-------------------+----------+----------+------------+------------+-----------------+
|order_id|            tanggal|  kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+----------+----------+------------+------------+-----------------+
|ORD-1019|2026-07-03 00:00:00|Elektronik|  Magelang|           9|       75000|         E-Wallet|
|ORD-1020|2026-07-22 00:00:00|Elektronik|  Semarang|           6|      150000|     Kartu Kredit|
|ORD-1028|2026-07-28 00:00:00|Elektronik|  Semarang|           8|      250000|    Transfer Bank|
|ORD-1040|2026-07-31 00:00:00|Elektronik| Purworejo|           6|      100000|         E-Wallet|
|ORD-1054|2026-07-25 00:00:00|Elektronik|Yogyakarta|           9|      500000|    Transfer Bank|
+--------+-------------------+----------+----------+------------+------------+-----------------+
only showing top 5 rows



In [11]:
from pyspark.sql.functions import sum as spark_sum, count, avg

# 1. Tambahkan kolom baru: total_pendapatan = unit_terjual * harga_satuan
df = df.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))

# 2. Meringkas performa penjualan per kota
ringkasan_kota = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
    count("order_id").alias("jumlah_transaksi"),
    avg("unit_terjual").alias("rata_rata_unit")
).orderBy(col("total_pendapatan").desc())

print("=== RINGKASAN PENJUALAN PER KOTA ===")
ringkasan_kota.show()

=== RINGKASAN PENJUALAN PER KOTA ===
+----------+----------------+----------------+-----------------+
|      kota|total_pendapatan|jumlah_transaksi|   rata_rata_unit|
+----------+----------------+----------------+-----------------+
|  Magelang|       109925000|             119| 5.07563025210084|
| Purworejo|       104675000|             122|4.737704918032787|
|Yogyakarta|       104625000|             129|5.248062015503876|
|  Semarang|        87200000|             118|4.771186440677966|
|      Solo|        83950000|             112|5.053571428571429|
+----------+----------------+----------------+-----------------+



# Integrasi Apache Spark + Hadoop

In [12]:
# 1. Menyiapkan folder direktori praktikum di HDFS menggunakan variabel dinamis $USER
!hdfs dfs -mkdir -p /user/$USER/pertemuan4

# 2. Mengunggah dataset ke direktori HDFS
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/$USER/pertemuan4/

# 3. Memverifikasi keberadaan berkas di klaster HDFS
!hdfs dfs -ls -h /user/$USER/pertemuan4

Found 1 items
-rw-r--r--   1 kaky supergroup     45.4 K 2026-09-16 21:38 /user/kaky/pertemuan4/data_transaksi_ecommerce.csv


In [13]:
import os

# Membaca CSV LANGSUNG dari klaster HDFS ke dalam Spark DataFrame
username = os.environ.get("USER", "kaky")
hdfs_path = f"hdfs://localhost:9000/user/{username}/pertemuan4/data_transaksi_ecommerce.csv"

print(f"Membaca dataset langsung dari URL: {hdfs_path}")
df_hdfs = spark.read.csv(hdfs_path, header=True, inferSchema=True)

print("Jumlah baris yang dimuat dari HDFS:", df_hdfs.count())
df_hdfs.show(5)

Membaca dataset langsung dari URL: hdfs://localhost:9000/user/kaky/pertemuan4/data_transaksi_ecommerce.csv
Jumlah baris yang dimuat dari HDFS: 600
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+
|ORD-1000|2026-07-07 00:00:00|Kesehatan & Kecan...|  Magelang|           9|      250000|         E-Wallet|
|ORD-1001|2026-07-20 00:00:00|   Makanan & Minuman|Yogyakarta|           9|       50000|     Kartu Kredit|
|ORD-1002|2026-07-29 00:00:00|             Fashion| Purworejo|           2|       50000|     Kartu Kredit|
|ORD-1003|2026-07-15 00:00:00|          Elektronik|      Solo|           3|       50000|         E-Wallet|
|ORD-1004|2026-07-11 00:00:00|          Elektronik|  Semarang|           5|      250000|    Transfer Ban

In [14]:
spark.stop()
print("SparkSession berhasil ditutup. Alokasi resource CPU & RAM telah dibebaskan.")

SparkSession berhasil ditutup. Alokasi resource CPU & RAM telah dibebaskan.
